# Richter VEP benchmark

Case/control benchmark of the variant effect predictions on the de novo variants (DNVs) of CHD probands and
controls (Richter et al. 2020): are high-scoring variants enriched in CHD probands (`is_CHD`)? Models: pretrained
AlphaGenome (whole-heart tracks), fine-tuned AlphaGenome and ChromBPNet (cardiomyocytes), next to CADD, phyloP and
the ENCODE4 cCRE signals. `is_CHD` is a label of the proband, not of the variant, so all cross-validation is grouped
by proband (`Blinded ID`).

1. **Per model**: AUPRC/AUROC, Mann-Whitney U box plots and AUPRC/AUROC per annotation stratum (cCREs, cardioid ATAC
   peaks and ABC enhancers, gene features, CHD gene lists).
2. **All models**: score correlations and AUROC per stratum.
3. **Systematic stratum search**: odds ratio of score-high variants (cutoff fit in grouped cross-validation) for
   every stratum and score, Benjamini-Hochberg FDR (`cardiomics_s2f.vep.enrichment`).
4. **Patient level**: are probands carrying >= 1 prioritised variant enriched among CHD probands?
5. **Random forest** on all scores and annotations, trained with grouped out-of-fold cross-validation
   (`cardiomics_s2f.vep.causality_classifier`).

**Inputs** (`config/paths.yaml`): `VCF_all_Richter_vars.csv` and `Richter2020_Supp_Tables.xlsx` in
`variant_data_dir` (`variant_datasets/richter_variant_dataset.ipynb`); the VEP outputs in `vep_dir` of
`alphagenome_api/04_vep.sh`, `alphagenome_ft/04_vep.sh` and `chrombpnet/03_vep.sh` on `VCF_all_Richter_vars.csv`;
the gene annotation (`gtf`, protein-coding genes, strand matching of the fine-tuned AlphaGenome RNA tracks).
Borzoi is not included (too slow for ~150,000 variants). Saving figures (to `figures_dir/richter_vep_benchmark`)
is commented out.

In [ ]:
import pandas as pd

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.eval.track_prediction import get_gtf
from cardiomics_s2f.vep.causality_classifier import plot_rf_feature_importances, train_grouped_oof_rf_classifier
from cardiomics_s2f.vep.enrichment import (
    build_stratum_grid,
    patient_level_enrichment,
    plot_top_strata_forest,
    systematic_patient_score_search,
    systematic_stratum_search,
)
from cardiomics_s2f.vep.plotting import (
    box_grid,
    plot_AUPRCs,
    plot_AUROCs,
    plot_score_correlation_clustermap,
    plot_stratified_auprc_heatmap,
)
from cardiomics_s2f.vep.processing import (
    format_alphagenome_ft_vep_top_gene,
    format_chrombpnet_vep,
    format_pretrained_alphagenome_vep_top_gene,
)

In [ ]:
paths = load_paths()
VCF_NAME = "VCF_all_Richter_vars"   # VEP input written by richter_variant_dataset.ipynb
VARIANT_DIR = paths.variant_data_dir
FIGURE_PATH = f"{paths.figures_dir}/richter_vep_benchmark/"

RICHTER_VCF = VARIANT_DIR / f"{VCF_NAME}.csv"
CHD_GENE_LISTS = VARIANT_DIR / "Richter2020_Supp_Tables.xlsx"

AG_PRE_RICHTER = paths.vep_dir / f"AG_VEP_out/{VCF_NAME}_naive_heart_tracks_aggr.csv"                # alphagenome_api/04_vep.sh
AG_FT_RICHTER = paths.vep_dir / f"AG_ft_VEP_out/VEP_AG_ft_fold_1_{VCF_NAME}.csv"                    # alphagenome_ft/04_vep.sh
CBPN_RICHTER = paths.vep_dir / f"chromBPNet_VEP_out/{VCF_NAME}_Cardiomyocytes.mean.variant_scores.tsv"  # chrombpnet/03_vep.sh

gtf = get_gtf(paths.gtf, chrom_filter="autosomesXY", gene_type_filter=["protein_coding"], feature=None)

In [ ]:
#fine-tuned AG: cardiomyocyte tracks, Cut&Tag days averaged, largest-effect strand-matched RNA gene per variant
ag_ft_richter = format_alphagenome_ft_vep_top_gene(pd.read_csv(AG_FT_RICHTER), gtf, cell_type="Cardiomyocytes")
ag_ft_richter

In [ ]:
#pretrained AG (whole-heart tracks): largest-effect RNA gene per variant, TF scores pivoted
ag_pre_richter = format_pretrained_alphagenome_vep_top_gene(pd.read_csv(AG_PRE_RICHTER))
ag_pre_richter

In [ ]:
richter_cbpn = pd.read_csv(CBPN_RICHTER, sep = "\t")
richter_cbpn = format_chrombpnet_vep(richter_cbpn)
richter_cbpn = richter_cbpn.drop(["abs_COUNTS_cPBN_raw_score", "COUNTS_cPBN_pval", "abs_COUNTS_cPBN_pval", "JSD_cPBN_pval", "abs_COUNTSxJSD_cPBN_raw_score", "COUNTSxJSD_pval", "abs_COUNTSxJSD_pval"], axis = 1)

In [ ]:
richter_meta = pd.read_csv(RICHTER_VCF) 
richter_meta = richter_meta.rename(columns={"Chromosome":'CHROM', "Pos":'POS', "Ref":'REF', "Alt":'ALT', "gene_name": "gtf_gene"}).drop(
    ['Start', 'End', 'cCRE_id', 'thickStart', 'thickEnd',"Cohort", "Simons Family ID"], axis = 1
)
richter_meta["cCRE_class"] = richter_meta["cCRE_class"].fillna("No cCRE")
richter_meta["gtf_feature"] = richter_meta["gtf_feature"].fillna("Intergenic")
richter_meta["gtf_gene"] = richter_meta["gtf_gene"].fillna("Intergenic")
richter_meta["gene_type"] = richter_meta["gene_type"].fillna("gene_type")
richter_meta["enhancer_regulatory_position"] = richter_meta["enhancer_regulatory_position"].fillna("No enhancer")
richter_meta[["DNase_maxZ","H3K4me3_maxZ","H3K27ac_maxZ","CTCF_maxZ", "phyloP_100way"]] = richter_meta[["DNase_maxZ","H3K4me3_maxZ","H3K27ac_maxZ","CTCF_maxZ", "phyloP_100way"]].fillna(0)

In [ ]:
richter_meta

In [ ]:
#merge all
ag_richter = pd.merge(ag_ft_richter, ag_pre_richter, on = "variant_id", how = "outer")
pred_richter = pd.merge(ag_richter, richter_cbpn, on = ["variant_id", 'CHROM', 'POS', 'REF', 'ALT'], how = "outer")

all_score_cols = list(pred_richter.columns[(pred_richter.columns.str.contains("_score")) & (pred_richter.columns != "abs_")])
abs_cols = [f"abs_{col}" for col in all_score_cols]
pred_richter[abs_cols] = pred_richter[all_score_cols].abs()

In [ ]:
richter = pd.merge(pred_richter, richter_meta,on = ["variant_id", 'CHROM', 'POS', 'REF', 'ALT'], how = "inner")
richter

In [ ]:
CHD_genes = pd.read_excel(
    CHD_GENE_LISTS,
    sheet_name='S5 CHD gene lists',
    usecols="A:F",     
    skiprows=1,         # Skips row 1, making row 2 your header
    nrows=4420
)
CHD_genes.head(3)

# AG pretrained

In [ ]:
ag_pre_scores = ["CADD_PHRED_score","phyloP_100way", *richter.columns[(richter.columns.str.contains("abs_")) & (richter.columns.str.contains("AG_pre_"))].sort_values()]
ag_pre_richter = richter.dropna(subset = ag_pre_scores)

In [ ]:
plot_AUPRCs(ag_pre_richter, true_label="is_CHD", score_columns=ag_pre_scores, n_cols = 8)

In [ ]:
plot_AUROCs(ag_pre_richter, true_label="is_CHD", score_columns=ag_pre_scores, n_cols = 7)

In [ ]:
box_grid(ag_pre_richter, x_col="is_CHD", y_cols=ag_pre_scores, n_cols = 7, alternative ="greater")

In [ ]:
#try stricitaion by differnt metrics
df = ag_pre_richter#[enhancer_richter.celltypes_in_enhancer.str.contains("Cardiomyocytes")]
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE dist. enhancer": df["cCRE_class"] == "Distal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "ATAC peak": df["is_cardioid_CA"].fillna(False),
    "ABC CM enhancer": df["in_cardioid_enhancer"].fillna(False) & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC dist. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="distal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Coding seq.": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
    "Intron": (df["gtf_feature"] == "intron"),
    "UTR": (df["gtf_feature"] == "UTR"),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_pre_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}


# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=ag_pre_scores, 
    strat_dict=my_stratifications,
    save_fig=False,
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
#try stricitaion by differnt metrics
df = ag_pre_richter[ag_pre_richter.is_cardioid_CA]
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE dist. enhancer": df["cCRE_class"] == "Distal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "ABC CM enhancer": df["in_cardioid_enhancer"].fillna(False) & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC dist. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="distal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Coding seq.": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
    "Intron": (df["gtf_feature"] == "intron"),
    "UTR": (df["gtf_feature"] == "UTR"),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_pre_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),    
}


# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=ag_pre_scores, 
    strat_dict=my_stratifications,
    save_fig=False,
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
df = ag_pre_richter[(ag_pre_richter.is_cardioid_CA) & (ag_pre_richter["cCRE_class"] =="CA-H3K4me3")]
plot_AUROCs(df, score_columns=ag_pre_scores, true_label="is_CHD", n_cols=7)                      

# AG finetuned

In [ ]:
ag_ft_scores = ["CADD_PHRED_score","phyloP_100way", *richter.columns[(richter.columns.str.contains("abs_")) & (richter.columns.str.contains("AG_ft_"))].sort_values()]
ag_ft_richter = richter.dropna(subset = ag_ft_scores)

In [ ]:
plot_AUPRCs(ag_ft_richter, true_label="is_CHD", score_columns=ag_ft_scores, n_cols = 7)

In [ ]:
plot_AUROCs(ag_ft_richter, true_label="is_CHD", score_columns=ag_ft_scores, n_cols = 7)

In [ ]:
#try stricitaion by differnt metrics
df = ag_ft_richter[ag_ft_richter.is_cardioid_CA]
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE dist. enhancer": df["cCRE_class"] == "Distal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "ATAC peak": df["is_cardioid_CA"].fillna(False),
    "ABC CM enhancer": df["in_cardioid_enhancer"].fillna(False) & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC dist. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="distal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Coding seq.": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
    "Intron": (df["gtf_feature"] == "intron"),
    "UTR": (df["gtf_feature"] == "UTR"),
    "Human CHD Gene": (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_ft_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_ft_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
    "Damaging DNV Gene": (df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"])))
}
# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=ag_ft_scores, 
    strat_dict=my_stratifications,
    save_fig=False,
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
box_grid(ag_ft_richter, x_col="is_CHD", y_cols=ag_ft_scores, n_cols = 7, alternative ="greater")

# chromBPNet

In [ ]:
cbpn_scores = ["CADD_PHRED_score","phyloP_100way", 'abs_COUNTS_cPBN_raw_score', 'JSD_cPBN_raw_score','abs_COUNTSxJSD_cPBN_raw_score']
cbpn_richter = richter.dropna(subset = cbpn_scores)

In [ ]:
plot_AUPRCs(cbpn_richter, true_label="is_CHD", score_columns=cbpn_scores, n_cols = 4)

In [ ]:
plot_AUROCs(cbpn_richter, true_label="is_CHD", score_columns=cbpn_scores, n_cols = 7)

In [ ]:
box_grid(cbpn_richter, x_col="is_CHD", y_cols=cbpn_scores, n_cols = 7, alternative ="greater")

In [ ]:
#try stricitaion by differnt metrics
df = cbpn_richter#[enhancer_richter.celltypes_in_enhancer.str.contains("Cardiomyocytes")]
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE dist. enhancer": df["cCRE_class"] == "Distal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "ATAC peak": df["is_cardioid_CA"].fillna(False),
    "ABC CM enhancer": df["in_cardioid_enhancer"].fillna(False) & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC dist. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="distal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Coding seq.": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
    "Intron": (df["gtf_feature"] == "intron"),
    "UTR": (df["gtf_feature"] == "UTR"),
}


# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=cbpn_scores, 
    strat_dict=my_stratifications,
    save_fig=False,
    metric="AUROC"
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

# All models comp

In [ ]:
all_scores = pd.Series([*cbpn_scores, *ag_ft_scores, *ag_pre_scores, 'DNase_maxZ',
       'H3K4me3_maxZ', 'H3K27ac_maxZ', 'CTCF_maxZ']).unique()
all_richter = richter.dropna(subset = all_scores)

In [ ]:
plot_score_correlation_clustermap(all_richter, all_scores)

In [ ]:
#try stricitaion by differnt metrics
df = all_richter#[enhancer_richter.celltypes_in_enhancer.str.contains("Cardiomyocytes")]
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE dist. enhancer": df["cCRE_class"] == "Distal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "ATAC peak": df["is_cardioid_CA"].fillna(False),
    "ABC CM enhancer": df["in_cardioid_enhancer"].fillna(False) & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC dist. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="distal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Coding seq.": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
    "Intron": (df["gtf_feature"] == "intron"),
    "UTR": (df["gtf_feature"] == "UTR"),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_pre_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}


# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=all_scores, 
    strat_dict=my_stratifications,
    alternate_score_names=[s.replace("abs_", "").replace("_raw_score", "").replace("_score", "").replace("_HIST", "").replace("_TF", "")  for s in all_scores],
    save_fig=False,
    metric="AUROC"
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
#try stricitaion by differnt metrics
df = all_richter[all_richter["cCRE_class"] == "Proximal enhancer"]
my_stratifications = {
    "All Vars": df.index,
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_pre_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}


# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=all_scores, 
    strat_dict=my_stratifications,
    alternate_score_names=[s.replace("abs_", "").replace("_raw_score", "").replace("_score", "").replace("_HIST", "").replace("_TF", "")  for s in all_scores],
    save_fig=False,
    metric = "AUROC"
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

In [ ]:
#try stricitaion by differnt metrics
df = all_richter[all_richter["cCRE_class"] == "Promoter"]
my_stratifications = {
    "All Vars": df.index,
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_pre_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
}


# Call the function
matrix_df = plot_stratified_auprc_heatmap(
    df=df, 
    true_label='is_CHD', 
    score_columns=all_scores, 
    strat_dict=my_stratifications,
    alternate_score_names=[s.replace("abs_", "").replace("_raw_score", "").replace("_score", "").replace("_HIST", "").replace("_TF", "")  for s in all_scores],
    save_fig=False,
    metric = "AUROC"
   # fig_path=FIGURE_PATH + 'AG_pretrained_Richter2020_enhancers_Causality_AUPRC_heatmap.pdf'
)

# Systemiatic stratification using OR

In [ ]:
top_scores = ['CADD_PHRED_score', 'phyloP_100way',
       'abs_COUNTSxJSD_cPBN_raw_score',
       'abs_AG_ft_ATAC_score', 'abs_AG_ft_H3K27ac_score', 'abs_AG_ft_H3K4me3_score',
        'abs_AG_ft_RNA_score','abs_AG_pre_RNA_raw_score',
       'abs_AG_pre_ATAC_raw_score',
       'abs_AG_pre_H3K27ac_HIST_raw_score',
 'abs_AG_pre_H3K27me3_HIST_raw_score',
 'abs_AG_pre_H3K36me3_HIST_raw_score',
 'abs_AG_pre_H3K4me1_HIST_raw_score',
 'abs_AG_pre_H3K4me3_HIST_raw_score',
 'abs_AG_pre_H3K9ac_HIST_raw_score',
 'abs_AG_pre_H3K9me3_HIST_raw_score',
       'abs_AG_pre_POLR2A_TF_raw_score',
]

In [ ]:
df = all_richter
my_stratifications = {
    "All Vars": df.index,
    "ATAC peak": (df.is_cardioid_CA.fillna(False)),
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
grid = build_stratum_grid(my_stratifications, max_depth=1)

results = systematic_stratum_search(
    df,
    score_cols=top_scores,
    stratum_grid=grid,
    cutoff_methods=["youden"], #richter all is roughly balanced
    group_col = "Blinded ID",
    min_n=100, n_splits=3,
)

In [ ]:
fig, ax = plot_top_strata_forest(results, top_n=10, rank_by="ci_low")

In [ ]:
df = all_richter[all_richter.is_cardioid_CA.fillna(False)]
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
grid = build_stratum_grid(my_stratifications, max_depth=1)

results_CA = systematic_stratum_search(
    df,
    score_cols=top_scores,
    stratum_grid=grid,
    cutoff_methods=["youden"], #richter all is roughly balanced
    group_col = "Blinded ID",
    min_n=100, n_splits=3,
)

In [ ]:
fig, ax = plot_top_strata_forest(results_CA, top_n=10, rank_by="ci_low")

In [ ]:
df = all_richter[all_richter["cCRE_class"] == "Proximal enhancer"]
my_stratifications = {
    "All Vars": df.index,
    "ATAC peak": (df.is_cardioid_CA.fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
grid = build_stratum_grid(my_stratifications, max_depth=1)

results_prx_enh = systematic_stratum_search(
    df,
    score_cols=top_scores,
    stratum_grid=grid,
    cutoff_methods=["youden"], #richter all is roughly balanced
    group_col = "Blinded ID",
    min_n=100, n_splits=3,
)

In [ ]:
fig, ax = plot_top_strata_forest(results_prx_enh, top_n=5, rank_by="ci_low", figsize=(8, 4.5))

In [ ]:
df = all_richter[all_richter["cCRE_class"] == "Promoter"]
my_stratifications = {
    "All Vars": df.index,
    "ATAC peak": (df.is_cardioid_CA.fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
grid = build_stratum_grid(my_stratifications, max_depth=1)

results_prom = systematic_stratum_search(
    df,
    score_cols=top_scores,
    stratum_grid=grid,
    cutoff_methods=["youden"], #richter all is roughly balanced
    group_col = "Blinded ID",
    min_n=100, n_splits=3,
)

In [ ]:
fig, ax = plot_top_strata_forest(results_prom, top_n=10, rank_by="ci_low")

# Confrim enrichment of CHD patients with these variants

In [ ]:
ag_scores = ['abs_AG_ft_ATAC_score', 'abs_AG_ft_H3K27ac_score', 'abs_AG_ft_H3K4me3_score',
        'abs_AG_ft_RNA_score','abs_AG_pre_RNA_raw_score',
       'abs_AG_pre_ATAC_raw_score',
       'abs_AG_pre_H3K27ac_HIST_raw_score',
 'abs_AG_pre_H3K27me3_HIST_raw_score',
 'abs_AG_pre_H3K36me3_HIST_raw_score',
 'abs_AG_pre_H3K4me1_HIST_raw_score',
 'abs_AG_pre_H3K4me3_HIST_raw_score',
 'abs_AG_pre_H3K9ac_HIST_raw_score',
]
ag_richter = richter.dropna(subset = ag_scores)

In [ ]:
df = ag_richter
scores = ['CADD_PHRED_score', 'phyloP_100way', 'abs_COUNTS_cPBN_raw_score',
       'JSD_cPBN_raw_score', 'abs_COUNTSxJSD_cPBN_raw_score',
       'abs_AG_ft_ATAC_score', 'abs_AG_ft_H3K27ac_score',
       'abs_AG_ft_H3K4me3_score', 'abs_AG_ft_RNA_score',
       'abs_AG_pre_ATAC_raw_score', 'abs_AG_pre_CTCF_TF_raw_score',
       'abs_AG_pre_H3K27ac_HIST_raw_score',
       'abs_AG_pre_H3K27me3_HIST_raw_score',
       'abs_AG_pre_H3K36me3_HIST_raw_score',
       'abs_AG_pre_H3K4me1_HIST_raw_score',
       'abs_AG_pre_H3K4me3_HIST_raw_score',
       'abs_AG_pre_H3K9ac_HIST_raw_score',
       'abs_AG_pre_H3K9me3_HIST_raw_score',
       'abs_AG_pre_POLR2A_TF_raw_score', 'abs_AG_pre_RNA_raw_score',
       'DNase_maxZ', 'H3K4me3_maxZ', 'H3K27ac_maxZ', 'CTCF_maxZ']
my_stratifications = {
    "All Vars": df.index,
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE prox. enhancer": df["cCRE_class"] == "Proximal enhancer",
    "cCRE dist. enhancer": df["cCRE_class"] == "Distal enhancer",
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    "ATAC peak": df["is_cardioid_CA"].fillna(False),
    "ABC CM enhancer": df["in_cardioid_enhancer"].fillna(False) & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC prox. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="proximal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "ABC dist. CM enhancer": (df["in_cardioid_enhancer"].fillna(False)) & (df["enhancer_regulatory_position"] =="distal") & (df["celltypes_in_enhancer"].str.contains("Cardiomyocytes").fillna(False)),
    "Coding seq.": (df["gtf_feature"] == "exon") | (df["gtf_feature"] =="start_codon") | (df["gtf_feature"] =="stop_codon") | (df["gtf_feature"] =="CDS"),
    "Intron": (df["gtf_feature"] == "intron"),
    "UTR": (df["gtf_feature"] == "UTR"),
    "Human CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Human CHD Genes (210)"]))),
    "Mouse CHD Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["Mouse CHD Genes (614)"]))),
    "HHE Gene": (df.AG_pre_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))) | (df.AG_ft_Gene.isin(set(CHD_genes["HHE Genes (4420)"]))),
    "Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
plot_stratified_auprc_heatmap(df, strat_dict=my_stratifications,
                              score_columns=scores,
                              alternate_score_names=[s.replace("abs_", "").replace("_raw_score", "").replace("_score", "").replace("_HIST", "").replace("_TF", "")  for s in scores],
                              true_label="is_CHD", metric="AUROC")

In [ ]:
df = ag_richter
my_stratifications = {
    "All Vars": df.index,
    "ATAC peak": (df.is_cardioid_CA.fillna(False)),
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE CA-H3K4me3": (df["cCRE_class"] =="CA-H3K4me3") ,
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    #"Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
grid = build_stratum_grid(my_stratifications, max_depth=1)

results_ag = systematic_stratum_search(
    df,
    score_cols=ag_scores,
    stratum_grid=grid,
    cutoff_methods=["f1", "youden"], #richter all is roughly balanced
    group_col = "Blinded ID",
    min_n=100, n_splits=2,
)

In [ ]:
fig, ax = plot_top_strata_forest(results_ag, top_n=10, rank_by="ci_low")

In [ ]:
plot_AUPRCs(ag_richter[ag_richter["cCRE_class"] == "TF"], score_columns=ag_scores, true_label="is_CHD", n_cols=12)

In [ ]:
plot_AUPRCs(ag_richter[ag_richter["cCRE_class"] == "Promoter"], score_columns=ag_scores, true_label="is_CHD", n_cols=12)

In [ ]:
plot_AUPRCs(ag_richter[ag_richter.is_cardioid_CA], score_columns=ag_scores, true_label="is_CHD", n_cols=12)

In [ ]:
results_ag.sort_values(by = "ci_low", ascending=False).head(15)

In [ ]:
# Cutoffs: cutoff_mean of the matching rows of results_ag (systematic_stratum_search above)
calib_dict ={}
calib_dict["TF"] = (ag_richter["cCRE_class"] == "CA-TF") &  (ag_richter.abs_AG_pre_RNA_raw_score >= 0.000199) & (ag_richter.abs_AG_pre_H3K4me1_HIST_raw_score >= 0.002198)# &  (ag_richter.abs_AG_ft_RNA_score >= 0.005230)
calib_dict["Promoter"] = (ag_richter["cCRE_class"] == "Promoter") & (ag_richter.abs_AG_pre_H3K4me1_HIST_raw_score >= 0.019205) & (ag_richter.abs_AG_ft_H3K4me3_score >= 0.014853) & (ag_richter.abs_AG_ft_RNA_score >= 0.035570)
calib_dict["Cardioid CA"] = (ag_richter.is_cardioid_CA) & (ag_richter.abs_AG_ft_ATAC_score >= 0.078492) & (ag_richter.abs_AG_pre_ATAC_raw_score >= 0.051790) & (ag_richter.abs_AG_pre_H3K27ac_HIST_raw_score >= 0.039788)
#calib_dict["all_calib"] = (ag_richter.abs_AG_pre_RNA_raw_score >= 0.002058) &  (ag_richter.abs_AG_pre_H3K27ac_HIST_raw_score >= 0.007354) &  (ag_richter.abs_AG_pre_H3K36me3_HIST_raw_score >= 0.003782)

calib_dict["Cardioid CA Promoter"] = calib_dict["Promoter"] & calib_dict["Cardioid CA"]
calib_dict["Cardioid CA TF"] = calib_dict["TF"] & calib_dict["Cardioid CA"]

In [ ]:
patient_results = patient_level_enrichment(ag_richter, calib_dict, label_col="is_CHD", group_col="Blinded ID",
                              min_carriers=5, alternative="greater", fdr_alpha=0.05)
patient_results

In [ ]:
fig, ax = plot_top_strata_forest(patient_results, top_n=5, rank_by="ci_low")

In [ ]:
#cehck how many variants this actually prioritizes

ag_richter[calib_dict["Cardioid CA"]].groupby("is_CHD").variant_id.nunique()

In [ ]:
#cehck how many variants this actually prioritizes
prio_CHD_vars = ag_richter[calib_dict["Cardioid CA"] & ag_richter.is_CHD].variant_id.unique()


richter_meta[richter_meta.variant_id.isin(set(prio_CHD_vars))].in_SDR007.value_counts()

In [ ]:
#now we run it systematically
df = ag_richter
my_stratifications = {
    "ATAC peak": (df.is_cardioid_CA.fillna(False)),
    "cCRE Promoter": (df["cCRE_class"] == "Promoter"),#
    "cCRE TF": (df["cCRE_class"] == "TF") | (df["cCRE_class"] =="CA-TF"),
    #"Damaging DNV Gene": (df.AG_pre_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))) |(df.AG_ft_Gene.isin(set(CHD_genes["Multiple WES damaging de novo Variant Genes (81)"]))),
}
grid = build_stratum_grid(my_stratifications, max_depth=1)

patient_score_results = systematic_patient_score_search(
    ag_richter,
    score_cols=['abs_AG_ft_ATAC_score',
                'abs_AG_ft_H3K27ac_score',
                 'abs_AG_ft_H3K4me3_score',
                 'abs_AG_pre_H3K27ac_HIST_raw_score',
 'abs_AG_pre_H3K36me3_HIST_raw_score',
 'abs_AG_pre_H3K4me1_HIST_raw_score',
 'abs_AG_pre_H3K9ac_HIST_raw_score'],
    cutoff_methods=["f1", "youden"],
    stratum_grid=grid, 
    group_col="Blinded ID", n_splits=2, min_carriers=5,
)

# RF calssification

Patient-grouped, stratified out-of-fold training (`train_grouped_oof_rf_classifier`): every variant is predicted
by a model that never saw its proband; per outer fold a randomised hyperparameter search with its own grouped inner
cross-validation. With the defaults (5 outer folds x 40 iterations x 5 inner folds) this fits ~1,000 forests;
reduce `n_iter` / the fold numbers for a quick run.

In [ ]:
numeric_cols = ['CADD_PHRED_score',
                'abs_COUNTSxJSD_cPBN_raw_score',
       'abs_AG_ft_ATAC_score',
        'abs_AG_ft_H3K27ac_score',
       'abs_AG_ft_H3K4me3_score', 'abs_AG_ft_RNA_score',
       'abs_AG_pre_ATAC_raw_score', 'abs_AG_pre_CTCF_TF_raw_score',
       'abs_AG_pre_H3K27ac_HIST_raw_score',
       'abs_AG_pre_H3K27me3_HIST_raw_score',
       'abs_AG_pre_H3K36me3_HIST_raw_score',
       'abs_AG_pre_H3K4me1_HIST_raw_score',
       'abs_AG_pre_H3K4me3_HIST_raw_score',
       'abs_AG_pre_H3K9ac_HIST_raw_score',
       'abs_AG_pre_H3K9me3_HIST_raw_score',
       'abs_AG_pre_POLR2A_TF_raw_score',
        'abs_AG_pre_RNA_raw_score',
      'H3K4me3_maxZ', 'H3K27ac_maxZ']

categorical_cols = [
    "cCRE_class", "gtf_feature",
    "gene_type", "is_cardioid_CA", "enhancer_regulatory_position",
    "is_cardioid_H3K27ac", "celltypes_in_enhancer", "in_cardioid_enhancer",
]

# columns whose NaN means "condition not met elsewhere" -- see docstring
conditional_cols = [
    "celltypes_in_enhancer", "is_cardioid_H3K27ac"
]

oof_df, importances_df, params_df = train_grouped_oof_rf_classifier(
    ag_richter,
    label_col="is_CHD",
    group_col="Blinded ID",  # patients: never split between training and test
    id_col="variant_id",
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    conditional_cols=conditional_cols,
    balance_cols=["CHROM", "variant_type"],
    outer_splits=5,
    inner_splits=5,
    n_iter=40,
    random_state=42,
)

In [ ]:
params_df

In [ ]:
#out-of-fold probabilities next to the single scores (rows in the order of ag_richter)
rf_richter = ag_richter.reset_index(drop=True).assign(predicted_proba=oof_df["oof_predicted_proba"].to_numpy())
assert (rf_richter["variant_id"].to_numpy() == oof_df["variant_id"].to_numpy()).all()
plot_AUROCs(rf_richter, true_label="is_CHD", score_columns=["predicted_proba", *numeric_cols], n_cols = 7)

In [ ]:
fig, imp_df = plot_rf_feature_importances(importances_df, numeric_cols, categorical_cols, top_n=25)